# Python Generators & Iterators Cheat Sheet

## 1. Iterators vs Iterables

- **Iterable** — anything you can loop over (`list`, `str`, `dict`, `range`)
- **Iterator** — an object that produces values one at a time via `next()`

An iterable becomes an iterator by calling `iter()` on it.

In [ ]:
my_list = [1, 2, 3]
it = iter(my_list)      # create an iterator

print(next(it))   # 1
print(next(it))   # 2
print(next(it))   # 3
# next(it)        # StopIteration — no more items

## 2. Generators — Functions that Yield

A **generator function** uses `yield` instead of `return`.

- Pauses at each `yield` and resumes on the next `next()` call
- Values are produced **lazily** — one at a time, using minimal memory

In [ ]:
def count_up(start, end):
    current = start
    while current <= end:
        yield current
        current += 1

gen = count_up(1, 5)
for n in gen:
    print(n)   # 1 2 3 4 5

In [ ]:
# Generators are memory efficient
# A list of 1 million numbers uses ~8 MB
# A generator uses almost nothing

def big_range(n):
    for i in range(n):
        yield i

gen = big_range(1_000_000)
print(next(gen))   # 0 — only one value in memory at a time

## 3. Generator Expressions

Like list comprehensions but with `()` — lazy, memory efficient.

In [ ]:
# List comprehension — builds the whole list in memory
squares_list = [x**2 for x in range(10)]

# Generator expression — lazy, one at a time
squares_gen = (x**2 for x in range(10))

print(list(squares_gen))   # [0, 1, 4, 9, 16, 25, 36, 49, 64, 81]

# Useful directly in functions that accept iterables
total = sum(x**2 for x in range(10))
print(total)   # 285

## 4. Practical Generator Examples

In [ ]:
# Infinite counter
def counter(start=0):
    n = start
    while True:
        yield n
        n += 1

c = counter()
print([next(c) for _ in range(5)])   # [0, 1, 2, 3, 4]

In [ ]:
# Fibonacci generator
def fibonacci():
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

fib = fibonacci()
print([next(fib) for _ in range(10)])  # [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]

## 5. Building a Custom Iterator Class

Implement `__iter__` and `__next__` to make any class iterable.

In [ ]:
class Countdown:
    def __init__(self, start):
        self.current = start

    def __iter__(self):
        return self

    def __next__(self):
        if self.current <= 0:
            raise StopIteration
        val = self.current
        self.current -= 1
        return val

for n in Countdown(5):
    print(n, end=" ")   # 5 4 3 2 1

## 6. itertools — Iterator Building Blocks

In [ ]:
import itertools

# chain — combine multiple iterables
result = list(itertools.chain([1,2], [3,4], [5]))
print(result)   # [1, 2, 3, 4, 5]

# islice — take first N from an infinite generator
fib = fibonacci()
first10 = list(itertools.islice(fib, 10))
print(first10)

# cycle — repeat an iterable
cycler = itertools.cycle(["A", "B", "C"])
print([next(cycler) for _ in range(7)])  # ['A', 'B', 'C', 'A', 'B', 'C', 'A']

---
## Practice Challenges

### Challenge 1: Prime Number Generator
Yield prime numbers indefinitely.

In [ ]:
def primes():
    def is_prime(n):
        if n < 2: return False
        for i in range(2, int(n**0.5)+1):
            if n % i == 0: return False
        return True
    n = 2
    while True:
        if is_prime(n):
            yield n
        n += 1

import itertools
print(list(itertools.islice(primes(), 10)))  # first 10 primes

### Challenge 2: Chunker
Write a generator that splits a list into chunks of size n.

In [ ]:
def chunker(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

data = list(range(1, 16))
for chunk in chunker(data, 4):
    print(chunk)

### Challenge 3: Lazy File Reader
Use a generator to read a large file line by line without loading it all into memory.

In [ ]:
from pathlib import Path

# Create a sample file
Path("large.txt").write_text("\n".join(f"Line {i}" for i in range(1, 21)))

def read_lines(path):
    with open(path) as f:
        for line in f:
            yield line.strip()

for line in read_lines("large.txt"):
    print(line)